In [30]:

import pandas as pd
import re
import ast
from html import escape
from IPython.display import display, HTML

In [31]:
df = pd.read_csv("tweede_kamer_data.csv", index_col=0)
df.head()

,title,year,ai_related,company_hits,body,matched_keywords_all,type
0,Investeren in Perspectief (Beleidsnota 2018),2018.0,yes,"['adyen', 'google', 'x']",Ook biedt de agenda kansen aan het bedrijfslev...,"['adyen', 'google', 'x', 'kunstmatige intellig...",beleidsnota
1,Nota Defensie Industrie Strategie,2018.0,yes,[],Nederland wil zelf aan militaire kennisontwikk...,"['ai', 'artificiële intelligentie', 'drones', ...",beleidsnota
2,Nationale Strategie Digitaal Erfgoed 2021-2024,2021.0,yes,['google'],Zo wordt in de vernieuwde strategie nu ook de ...,"['google', 'ai', 'algoritmes', 'artificiële in...",beleidsnota
3,Beslisnota's bij de Kamerbrief over toekomstig...,2021.0,yes,['x'],25 januari\n37 9-2-2023 Nota-StasGB-Memo box 3...,"['x', 'ai']",beleidsnota
4,Beslisnota bij Kamerbrief over aanpak belastin...,2021.0,yes,['x'],Dit betreft de nota’s in de onderstaande tabel...,"['x', 'ai']",beleidsnota


In [32]:
import ast

def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return []
    return []

df['matched_keywords_all'] = df.apply(
    lambda row: to_list(row['company_hits']) + to_list(row['matched_keywords_all']),
    axis=1
)
df['matched_keywords_all'].iloc[0]

['adyen', 'google', 'x', 'adyen', 'google', 'x', 'kunstmatige intelligentie']

In [33]:


def inspect_ai_related(df, body='body',  num_samples=2, random_state=42):
    """
    Displays random samples with highlighted *matched* keywords.
    Uses df['matched_keywords_all'] per row instead of a global keyword list.
    """
    
    # --- sanity checks ---
    if 'ai_related' not in df.columns:
        raise KeyError("DataFrame must contain an 'ai_related' column.")
    if not {'title', body}.issubset(df.columns):
        missing = {'title', body} - set(df.columns)
        raise KeyError(f"Missing required column(s): {missing}")
    if 'matched_keywords_all' not in df.columns:
        raise KeyError("DataFrame must contain 'matched_keywords_all'.")

    # --- sample ---
    n = min(num_samples, len(df))
    samples = df.sample(n=n, random_state=random_state)

  
    

    # --- internal highlight function ---
    def highlight_keywords(text, keywords):
        if pd.isna(text):
            return ""
        s = str(text)

        if not isinstance(keywords, (set, list)):
            raise ValueError("Keywords must be a set or list")
        kws = {str(w).strip().lower() for w in keywords if str(w).strip()}
        if not kws:
            return s

        # sort by length to avoid partial overshadowing
        ordered = sorted(kws, key=len, reverse=True)

        # custom boundary: match even inside hyphenated words
        def make_pattern(word):
            return rf'(?<![A-Za-z0-9]){re.escape(word)}(?![A-Za-z0-9])'

        combined = "|".join(make_pattern(w) for w in ordered)
        regex = re.compile(combined, flags=re.IGNORECASE)

        def repl(m):
            kw = m.group(0)
            return f'<span style="font-size:1.5em; font-weight:bold; color:green;">{escape(kw)}</span>'

        return regex.sub(repl, s)

    # --- display samples ---
    for idx, row in samples.iterrows():
        
        ai_val = row['ai_related']
        title = row['title']
        body_text = row[body]  # ✅ 'body' parameter stays intact

         # --- use the actually matched keywords in this row ---
        matched_keywords = row.get('matched_keywords_all', [])
        if isinstance(matched_keywords, (list, set, tuple)):
            row_terms = {str(x).lower() for x in matched_keywords}
        else:
            row_terms = {str(matched_keywords).lower()} if matched_keywords else set()

        # --- highlight ---
        highlighted_title = highlight_keywords(title, row_terms)
        highlighted_body  = highlight_keywords(body_text, row_terms)

        # --- header ---
        header_html = (
            f'<div style="margin:0.5em 0;">'
            f'<strong>Index:</strong> {idx} &nbsp; | &nbsp; '
            f'<strong>ai_related:</strong> {ai_val}'
        )
        if matched_keywords:
            header_html += f'<br><strong>matched keywords:</strong> {matched_keywords}'
        header_html += '</div>'

        # --- display ---
        display(HTML(header_html))
        display(HTML(f'<h3 style="margin:0.2em 0;">{highlighted_title}</h3>'))
        display(HTML(f'<div style="line-height:1.5;">{highlighted_body}</div>'))
        display(HTML('<hr>'))

    print(f"Displayed {n} random articles (with row-specific matched keywords highlighted).")
    return samples
    

In [34]:
# # program = "all" to display from all programs, or specify a program like "jinek"
# inspect_ai_related(df, body = 'body', num_samples=20, random_state=42)

In [39]:
# check how many hits of 'x' in matched keywords_all
keyword_to_check = 'x'
count = df['matched_keywords_all'].apply(lambda kws: keyword_to_check in kws if isinstance(kws, (list, set)) else False).sum()
print(f"Number of rows where '{keyword_to_check}' is in matched_keywords_all: {count}")

# check how often 'x' is the only keyword in matched_keywords_all
count_only = df['matched_keywords_all'].apply(lambda kws: isinstance(kws, (list, set)) and len(kws) == 1 and keyword_to_check in kws).sum()
print(f"Number of rows where '{keyword_to_check}' is the only keyword in matched_keywords_all: {count_only}")

Number of rows where 'x' is in matched_keywords_all: 348
Number of rows where 'x' is the only keyword in matched_keywords_all: 0
